In [1]:
from z3 import *
from abc import ABC, abstractmethod

In [22]:
class LanguageE:
    def __init__(self):
        # Core fields required by the engine
        self.axioms: list[ExprRef] = []
        self.solver: Solver = Solver()
        
        # Declare uninterpreted sorts for Euclid's elements
        self.PointSort: SortRef = DeclareSort('Point')
        self.LineSort: SortRef = DeclareSort('Line')
        self.CircleSort: SortRef = DeclareSort('Circle')
        
        # Constants
        self.RightAngle: ExprRef = Real('RightAngle')
        
        # --- Predicates / Relations (Return BoolSort) ---
        # Function('Name', InputSort1, InputSort2, ..., OutputSort)
        self.Between = Function('Between', self.PointSort, self.PointSort, self.PointSort, BoolSort())
        self.OnLine = Function('OnLine', self.PointSort, self.LineSort, BoolSort())
        self.OnCircle = Function('OnCircle', self.PointSort, self.CircleSort, BoolSort())
        self.Inside = Function('Inside', self.PointSort, self.CircleSort, BoolSort())
        self.Center = Function('Center', self.PointSort, self.CircleSort, BoolSort())
        self.SameSide = Function('SameSide', self.PointSort, self.PointSort, self.LineSort, BoolSort())
        
        # Intersection Predicates
        self.Intersectsll = Function('Intersectsll', self.LineSort, self.LineSort, BoolSort())
        self.Intersectslc = Function('Intersectslc', self.LineSort, self.CircleSort, BoolSort())
        self.Intersectscc = Function('Intersectscc', self.CircleSort, self.CircleSort, BoolSort())
        
        # --- Magnitudes / Metrics (Return RealSort) ---
        self.Segment = Function('Segment', self.PointSort, self.PointSort, RealSort())
        self.Angle = Function('Angle', self.PointSort, self.PointSort, self.PointSort, RealSort())
        self.Area = Function('Area', self.PointSort, self.PointSort, self.PointSort, RealSort())

In [31]:
class Proof:
    def __init__(self):
        self.points: set[Point] = set()
        self.lines: set[Line] = set()
        self.circles: set[Circle] = set()
        
        self.assumptions: set[ExprRef] = set()
        self.conclusions: set[ExprRef] = set()
        self.isProved: bool = False

        self.language: LanguageE = LanguageE()
        self.solver = self.language.solver
        
    def construct(self, constructableObject):
        pass
    
    def hence(self, exprRef):
        pass
    
    def asserts(self, proof):
        pass
    
    def qed(self):
        pass

In [34]:
class ConstructableObject(ABC):
    def __init__(self, label: str, sort: str):
        # Fields straight from the rojas_thesis.pdf object model:
        self.z3Expr: ExprRef = None         # The underlying Z3 expression representation
        self.label: str = label             # e.g., "a", "b", "L"
        self.sort: str = sort               # "Point", "Line", or "Circle"[cite: 1]
        self.prereqs: list[ExprRef] = []    # Language E preconditions for construction[cite: 1]
        self.conclusions: list[ExprRef] = []# Language E post-conditions/conclusions[cite: 1]
        self.isDistinct: bool = True        # Unique object flag[cite: 1]

    @abstractmethod
    def equals(self, obj: object) -> bool: # Maps to + equals(Obj) in the UML[cite: 1]
        """Every geometric object must implement its own equality logic."""
        pass

In [37]:
class Circle(ConstructableObject):
    def __init__(self, label: str, language: LanguageE):
        # 1. Initialize the abstract parent fields with the sort "Circle"
        super().__init__(label, "Circle")
        
        # 2. Create the actual Z3 constant object using the CircleSort we defined
        self.z3Expr = Const(label, language.CircleSort)
        
    def centerThrough(self, point1: 'Point', point2: 'Point', language: LanguageE):
        """
        Defines a circle with center at point1, passing through point2.
        """
        # Prerequisites: To draw a valid circle, the center and radius point cannot be identical[cite: 1]
        self.prereqs = [point1.z3Expr != point2.z3Expr]
        
        # Conclusions: Drawing this circle introduces two facts into Language E[cite: 1]:
        # 1. point1 is the center of this circle[cite: 1]
        # 2. point2 lies on the boundary of this circle[cite: 1]
        self.conclusions = [
            language.Center(point1.z3Expr, self.z3Expr),
            language.OnCircle(point2.z3Expr, self.z3Expr)
        ]
        
    def equals(self, obj: object) -> bool:
        # Fulfilling the abstract method obligation[cite: 1]
        if isinstance(obj, Circle):
            return self.label == obj.label
        return False

In [38]:
class Line(ConstructableObject):
    def __init__(self, label: str, language: LanguageE):
        # 1. Initialize the abstract parent fields with the sort "Line"
        super().__init__(label, "Line")
        
        # 2. Create the actual Z3 constant object using the LineSort defined in LanguageE
        self.z3Expr = Const(label, language.LineSort)
        
    def through(self, point1: 'Point', point2: 'Point', language: LanguageE):
        """
        Defines a straight line that passes through two distinct points.
        """
        # Prerequisites: To construct a unique line, the two points must not be identical
        self.prereqs = [point1.z3Expr != point2.z3Expr]
        
        # Conclusions: Drawing this line adds two facts to the proof engine:
        # 1. point1 is on this line
        # 2. point2 is on this line[cite: 1]
        self.conclusions = [
            language.OnLine(point1.z3Expr, self.z3Expr),
            language.OnLine(point2.z3Expr, self.z3Expr)
        ]
        
    def equals(self, obj: object) -> bool:
        # Fulfilling the abstract method obligation from ConstructableObject[cite: 1]
        if isinstance(obj, Line):
            return self.label == obj.label
        return False

In [39]:
from z3 import Const

class Point(ConstructableObject):
    def __init__(self, label: str, language: LanguageE):
        # 1. Initialize abstract parent fields with the sort "Point"
        super().__init__(label, "Point")
        
        # 2. Create the Z3 constant object using PointSort from LanguageE
        self.z3Expr = Const(label, language.PointSort)
        
    def onLine(self, line: 'Line', language: LanguageE):
        """Constructs an arbitrary point on a given line."""
        self.prereqs = []
        self.conclusions = [language.OnLine(self.z3Expr, line.z3Expr)]
        
    def onCircle(self, circle: 'Circle', language: LanguageE):
        """Constructs an arbitrary point on the boundary of a circle.[cite: 1]"""
        self.prereqs = []
        self.conclusions = [language.OnCircle(self.z3Expr, circle.z3Expr)]
        
    def between(self, point1: 'Point', point2: 'Point', language: LanguageE):
        """Constructs a point collinear to and strictly between two other points.[cite: 1]"""
        # Prerequisite: The two bounding points cannot be identical[cite: 1]
        self.prereqs = [point1.z3Expr != point2.z3Expr]
        self.conclusions = [language.Between(point1.z3Expr, self.z3Expr, point2.z3Expr)]
        
    def sameside(self, point1: 'Point', line: 'Line', language: LanguageE):
        """Constructs a point on the same side of a line as another given point.[cite: 1]"""
        self.prereqs = []
        self.conclusions = [language.SameSide(self.z3Expr, point1.z3Expr, line.z3Expr)]
        
    def opposite(self, point1: 'Point', line: 'Line', language: LanguageE):
        """Constructs a point on the opposite side of a line from a given point.[cite: 1]"""
        self.prereqs = []
        # In Language E, opposite side is logically the negation of SameSide[cite: 1]
        from z3 import Not
        self.conclusions = [Not(language.SameSide(self.z3Expr, point1.z3Expr, line.z3Expr))]
        
    def inside(self, circle: 'Circle', language: LanguageE):
        """Constructs a point located inside the interior area of a circle.[cite: 1]"""
        self.prereqs = []
        self.conclusions = [language.Inside(self.z3Expr, circle.z3Expr)]
        
    def outside(self, circle: 'Circle', language: LanguageE):
        """Constructs a point located strictly outside a circle.[cite: 1]"""
        self.prereqs = []
        from z3 import Not
        self.conclusions = [
            Not(language.Inside(self.z3Expr, circle.z3Expr)),
            Not(language.OnCircle(self.z3Expr, circle.z3Expr))
        ]
        
    def intersectsLines(self, line1: 'Line', line2: 'Line', language: LanguageE):
        """Constructs a point where two lines cross each other.[cite: 1]"""
        # Prerequisite: The two lines must actually intersect[cite: 1]
        self.prereqs = [language.Intersectsll(line1.z3Expr, line2.z3Expr)]
        self.conclusions = [
            language.OnLine(self.z3Expr, line1.z3Expr),
            language.OnLine(self.z3Expr, line2.z3Expr)
        ]
        
    def intersectsCircleLine(self, circle: 'Circle', line: 'Line', language: LanguageE):
        """Constructs a point where a circle boundary intersects a straight line.[cite: 1]"""
        # Prerequisite: The line and circle must intersect[cite: 1]
        self.prereqs = [language.Intersectslc(line.z3Expr, circle.z3Expr)]
        self.conclusions = [
            language.OnCircle(self.z3Expr, circle.z3Expr),
            language.OnLine(self.z3Expr, line.z3Expr)
        ]
        
    def intersectsCircleCircle(self, circle1: 'Circle', circle2: 'Circle', language: LanguageE):
        """Constructs a point at the intersection point of two circle boundaries.[cite: 1]"""
        # Prerequisite: Both circles must intersect[cite: 1]
        self.prereqs = [language.Intersectscc(circle1.z3Expr, circle2.z3Expr)]
        self.conclusions = [
            language.OnCircle(self.z3Expr, circle1.z3Expr),
            language.OnCircle(self.z3Expr, circle2.z3Expr)
        ]
        
    def equals(self, obj: object) -> bool:
        # Fulfilling the abstract method obligation[cite: 1]
        if isinstance(obj, Point):
            return self.label == obj.label
        return False

In [101]:
from z3 import *
# Define System E Language
PointSort = DeclareSort('Point')
LineSort = DeclareSort('Line')
CircleSort = DeclareSort('Circle')
RightAngle = RealVal(90)
bet = Function('bet', PointSort, PointSort, PointSort, BoolSort())
onLine = Function('onLine', PointSort, LineSort, BoolSort())
onCircle = Function('onCircle', PointSort, CircleSort, BoolSort())
inside = Function('inside', PointSort, CircleSort, BoolSort())
center = Function('center', PointSort, CircleSort, BoolSort())
sameside = Function('sameside', PointSort, PointSort, LineSort, BoolSort())
# Intersection Predicates
interLL = Function('interLL', LineSort, LineSort, BoolSort())
interLC = Function('interLC', LineSort, CircleSort, BoolSort())
interCC = Function('interCC', CircleSort, CircleSort, BoolSort())
# --- Magnitudes / Metrics (Return RealSort) ---
Segment = Function('Segment', PointSort, PointSort, RealSort())
Angle = Function('Angle', PointSort, PointSort, PointSort, RealSort())
Area = Function('Area', PointSort, PointSort, PointSort, RealSort())

A, B, C, D, E,F,G,H,I,J,K,L,M,N,P,Q,R,S = Consts('A B C D E F G H I J K L M N P Q R S', LineSort)
a,b,c,d,e,f,g = Consts('a b c d e f g', PointSort)
α,β,γ,ω = Consts('α β γ ω', CircleSort)
## Const of Sort instance Initialiser
Point = lambda name: Const(name, PointSort)
Line = lambda name: Const(name, LineSort)
Circle = lambda name: Const(name, CircleSort)
## Axioms
axioms = { ## General Rules
'G1': ForAll([a,b,L,M], Implies( And(a != b, onLine(a,L), onLine(a,M), onLine(b,L), onLine(b,M)), L == M )), #兩點皆在線和線，則線線本同
'G2': ForAll([a,b,α,β], Implies( And(center(a,α), center(b,β)) , a == b )),  #點點忠圓則本同
'G3': ForAll([a,α], Implies( center(a,α) , inside(a,α) )),           #圓心入圓
'G4': ForAll([a,α], Implies( inside(a,α) , Not(onCircle(a,α)) )),    #圓內點不在圓
 ## Betweenness
'B1': ForAll([a,b,c], Implies( bet(a,b,c) , And( bet(c,b,a), a!=b, a!=c, Not(bet(b,a,c)) )  )),  #排則三點各異，首尾可換但首間不可換
'B2': ForAll([a,b,c], Implies( And(bet(a,b,c), onLine(a,L), onLine(b,L)), onLine(c,L)  )),   #線過排首兩點，則也過尾點
'B3': ForAll([a,b,c], Implies( And(bet(a,b,c), onLine(a,L), onLine(c,L)), onLine(b,L)  )),   #線過排首尾點，則也過間點
'B4': ForAll([a,b,c,d], Implies( And(bet(a,b,c), bet(a,d,b)), bet(a,d,c) )),    #排首兩之間點在首尾之間
'B5': ForAll([a,b,c,d], Implies( And(bet(a,b,c), bet(b,c,d)), bet(a,b,d) )),  #排尾兩是另排首兩，則間點在排首另排尾之間
'B6': ForAll([a,b,c,L], Implies( And(Distinct(a,b,c), onLine(a,L), onLine(b,L), onLine(c,L)),    #三點在線，必有一間點焉
                               And(AtLeast(bet(a,b,c),bet(b,a,c),bet(a,c,b), 1), AtMost(bet(a,b,c),bet(b,a,c),bet(a,c,b), 1)) )),
'B7': ForAll([a,b,L], Implies( And(bet(a,b,c), bet(a,b,d)), Not(bet(c,b,d)) )),   #兩排共首兩，則間點不在尾們之間
 ## Samesideness
'S1': ForAll([a,L], Implies( Not(onLine(a,L)), sameside(a,a,L) )),  #不在線點與己同邊
'S2': ForAll([a,b,L], Implies( sameside(a,b,L), sameside(b,a,L) )),   #點點同線邊可交換
'S3': ForAll([a,b,L], Implies( sameside(a,b,L), Not(onLine(a,L)) )),  #點點同線邊不在線
'S4': ForAll([a,b,c,L], Implies( And(sameside(a,b,L), sameside(a,c,L)), sameside(b,c,L) )),  #點點共點同線邊，則點點也同線邊
'S5': ForAll([a,b,c,L], Implies( And(Not(onLine(a,L)), Not(onLine(b,L)), Not(onLine(c,L)), Not(sameside(a,b,L)) ), 
                                 Xor(sameside(a,c,L),sameside(b,c,L)) )),   #點點點選線邊站，點點各一邊，則餘點必與點同邊
 ## Pasch axioms
'P1': ForAll([a,b,c,L], Implies( And(bet(a,b,c), sameside(a,c,L)), sameside(a,b,L)  )),   #排首尾同線邊，則首兩也同邊
'P2': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(a,L), Not(onLine(b,L))), sameside(b,c,L)  )),  #排首在線且間點不在線，則間尾同邊
'P3': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(b,L)), Not(sameside(a,c,L)) )),   #排間點在線，則首尾不同邊
'P4': ForAll([a,b,c,L,M], Implies( And(L!=M, onLine(b,L), onLine(b,M), onLine(a,M), onLine(c,M), Distinct(a,b,c), Not(sameside(a,c,L))), 
                                   bet(a,b,c) )),   #兩線有交點，另兩點皆在甲線且不同乙線邊，則交點為間點
## SKIPPED Triple incidence axioms FOR NOW
 ## Circles rules
'C1': ForAll([a,b,c,L,α], Implies( And(onLine(a,L),onLine(b,L),onLine(c,L), inside(a,α),onCircle(b,α),onCircle(c,α), b!=c), 
                                   bet(b,a,c) )),   #三點在線，其一入圓，餘兩點在圓，則入圓者在兩點之間
'C2': ForAll([a,b,c,α], Implies( And(Or(inside(a,α),onCircle(a,α)), Or(inside(b,α),onCircle(b,α)), bet(a,c,b)  ), 
                                   inside(c,α) )),   #排首尾皆在圓盤，則間點入圓
'C3': ForAll([a,b,c,α], Implies( And(Or(inside(a,α),onCircle(a,α)), Not(inside(c,α)), bet(a,c,b)  ), 
                                   And(Not(inside(b,α)), Not(onCircle(b,α))) )),   #排首在圓盤，間點不入圓，則尾出圓
'C4': ForAll([a,b,c,d,L,α,β], Implies( And(α!=β, c!=d, onCircle(c,α), onCircle(d,α), onCircle(c,β), onCircle(d,β), 
                                         center(a,α), center(b,β), onLine(a,L), onLine(b,L)   ), 
                                   Not(sameside(c,d,L)) )),    #兩圓交之兩點不同過各圓心之線邊
 ## Intersection rules
'I1': ForAll([a,b,L,M], Implies( And(Not(sameside(a,b,L)),Not(onLine(a,L)),Not(onLine(b,L)), onLine(a,M), onLine(b,M)  ), 
                                   interLL(L,M) )),   #隔開兩點之線必交過兩點之線
'I2': ForAll([a,b,L,α], Implies( And(Or(inside(a,α),onCircle(a,α)), Or(inside(b,α),onCircle(b,α)), 
                                     Not(sameside(a,b,L)),Not(onLine(a,L)),Not(onLine(b,L)) ), 
                                   interLC(L,α) )),   #隔開圓盤兩點之線交圓
'I3': ForAll([a,L,α], Implies( And(inside(a,α), onLine(a,L),   ), interLC(L,α) )),   #過圓內點之線交圓
'I4': ForAll([a,b,α,β], Implies( And(onCircle(a,α), Or(inside(b,α),onCircle(b,α)), inside(a,β), Not(Or(inside(b,β),onCircle(b,β))) ), 
                                   interCC(α,β) )),   #過圓內點並包含圓外點之另圓交圓
'I5': ForAll([a,b,α,β], Implies( And(onCircle(a,α), inside(b,α), inside(a,β), onCircle(b,β) ), 
                                   interCC(α,β) )),   #兩圓互過對方內點則相交
 ## Metric rules
'M1': ForAll([a,b], (Segment(a,b) == RealVal(0)) == (a == b) ),   #段長為零即端本同
'M2': ForAll([a,b], (Segment(a,b) >= RealVal(0)) ),  #段長非負
'M3': ForAll([a,b], Segment(a,b) == Segment(b,a) ),  #段端可互換
'M4': ForAll([a,b,c], Implies(And(a!=b, b!=c), Angle(a,b,c)==Angle(c,b,a)) ),  #角端不同頂，角端可互換
'M5': ForAll([a,b,c], And(RealVal(0) <= Angle(a,b,c), Angle(a,b,c) <= RealVal(180)) ),  #角度介乎零至一八零
'M6': ForAll([a,b], Area(a,a,b) == 0 ),  #兩角形面積為零
'M7': ForAll([a,b,c], Area(a,b,c) >= 0 ),   #面積非負
'M8': ForAll([a,b,c], Area(a,b,c) == Area(c,a,b) == Area(a,c,b) ),  #置換三角形頂點不變面積
'M9': ForAll([a,b,c], Implies(And(Segment(a,b)==Segment(d,e),Segment(b,c)==Segment(e,f), Segment(c,a)==Segment(f,d),
                                 Angle(a,b,c)==Angle(d,e,f), Angle(b,c,a)==Angle(e,f,d), Angle(c,a,b)==Angle(f,d,e) ),
                             Area(a,b,c) == Area(d,e,f) )),    #兩三角形對應邊角相等，則面積相等
    
    # 'MT': ForAll([P, Q], Implies(And(Valid(imp(P, Q)), Valid(neg(Q))), Valid(neg(P))) ),
    # # HS: p->q, q->r |= p->r
    # 'HS': ForAll([P,Q,R], Implies(And(Valid(imp(P,Q)), Valid(imp(Q,R)) ), Valid(imp(P,R)) ) ),
    # 'DS': ForAll([P,Q], Implies(And(Valid(lor(P, Q)), Valid(neg(P))), Valid(Q))  ),   # DS: Disjunctive syllogism
    # 'CD': ForAll([P,Q,R,S], Implies(And(Valid(lor(imp(P, Q), imp(R, S))), Valid(lor(P, R))), Valid(lor(Q, S)))),# CD: Constructive Dilemma
    # 'Abs': ForAll([P,Q], Implies(And(Valid(imp(P, Q))), Valid(imp(P, land(P, Q))))  ),  # Abs: Absorption
    # 'Simp': ForAll([P,Q], Implies(And(Valid(land(P, Q))), Valid(P))  ),  # Simp: Simplification
    # 'Conj': ForAll([P,Q], Implies(And(Valid(P), Valid(Q)), Valid(land(P, Q))) ),  # Conj: Conjunction
    # 'Add': ForAll([P,Q], Implies(And(Valid(P)), Valid(lor(P, Q))) )   # Add: Addition
}


In [57]:
class Proof:
    def __init__(self, premises, goal, solver: Solver = None):
        if solver is None:
            solver = Solver(); solver.set("timeout", 500)
        self.solver = solver
        self.axioms = axioms
        self.z3_vars = {}    # Store Z3 variables, indexed by str
        self.context: [Bool] = [ None ]    # Stores premises and any valid formulas proved
        self.goal = 'undefined'
        self.goalrepr = 'undefined'
        self.loadContextAndGoal(premises, goal)

    def to_z3(self, node):  # Helper to recursively compiles a TreeNode directly into a real Z3 expression.
        if node is None: return None  
        # Leaf Node: It's an atom variable (J, K, L, etc.)
        if node.left is None and node.right is None:
            if node.value not in self.z3_vars:
                # Dynamically create it as a Z3 constant if we haven't seen it yet
                self.z3_vars[node.value] = Const(node.value, Formula)
            return self.z3_vars[node.value]
        # Unary Operator: Negation
        if node.value == '¬':
            return neg(self.to_z3(node.left))
        # Binary Operators: Convert left and right child subtrees first
        left_z3 = self.to_z3(node.left)
        right_z3 = self.to_z3(node.right)
        if node.value == '∧':   return land(left_z3, right_z3)
        elif node.value == '∨': return lor(left_z3, right_z3)
        elif node.value == '→': return imp(left_z3, right_z3)
        raise ValueError(f"Unknown operator in AST: {node.value}")

    def replaceASCII(self, template):  # Helper to turn ASCII operators into logical symbols
        return template.replace('~','¬').replace('&','∧').replace('|','∨').replace('->','→').replace(':','⊢')

    def loadContextAndGoal(self, premises:list[ExprRef], goal:ExprRef):   # Called at init
        self.context = premises
        self.solver.add(premises)
        self.goal = goal
        for i,t in enumerate(premises):    # Print premises and conclusion for view
            print(f"{i+1}. {t}", end='\n' if i< len(premises)-1 else '')
        print(f'    / {self.goal}')
        print('-'*20)
        return
        # ========= Old stuff ==========
        template = self.replaceASCII(template)
        pc = template.split('⊢')   # seperate into premises, conclusion
        pc = [ t.strip() for t in pc ]    # Trim trailing spaces
        premises = pc[0].split(',')     # Split premise text into list
        for i,t in enumerate(premises):    # Print premises and conclusion for view
            print(f"{i+1}. {t}", end='\n' if i< len(premises)-1 else '')
        self.goalrepr = pc[1]
        print(f'    / {self.goalrepr}')
        print('-'*20)
        # Load premise into self.context
        for prem in premises:
            ast_tree = parse(prem)
            z3_expr = Valid(self.to_z3(ast_tree))
            self.context.append( z3_expr )
            # print(f"Loaded Premise into Z3: {z3_expr}")
        # Load goal text into self.goal
        goal_tree = parse(pc[1])
        self.goal = Valid(self.to_z3(goal_tree))
        #print(f"Loaded Goal into Z3: {self.goal}")

    def have(self, fmla: ExprRef, rule: str):    # Infer step
        self.solver.push()    # Make checkpoint
        try:
            self.solver.add( axioms[rule] )   # Only allow this axiom in solver
        except KeyError:
            print(f"In request #{len(self.context)+1} have({fmla}, {rule}):", end=' ')
            print("Invalid reason!") 
            self.solver.pop(); return
        if fmla in self.context:
            print(f"In request #{len(self.context)+1} have({fmla}, {rule}):", end=' ')
            print(f"You have proved {fmla} before. Try again.")
            self.solver.pop(); return
        self.solver.add( Not(fmla) )
        result = self.solver.check()
        
        if result == unsat:    # Successful deduction step
            self.context.append( fmla )
            print(f"{len(self.context)}. {fmla}\t[{rule}]")
        else:
            print(f"In request #{len(self.context)+1} have({fmla}, {rule}):", end=' ')
            print(f"Sorry, can't infer {fmla} from {rule}"); return
        self.solver.pop();   # Remove the axiom used from solver 
        self.solver.add(fmla); return

    def qed(self):  # Check if any formula in self.context is the goal
        for fmla in self.context:
            if fmla == self.goal:
                print(f"🔥 Success! You proved {self.goal} is true."); return
        print(f"❌ You haven't proved {self.goal} yet.")
        return

In [80]:
pc = Proof([bet(a,b,c), bet(b,a,d) ], bet(c,b,d))    # test
pc.have(bet(c,b,a), 'B1')
pc.have(bet(c,b,d), 'B5')
pc.qed()
# pc.have('P', 'MP',2,5)
# pc.have('~Q', 'MP',1,6)
# pc.qed()

1. bet(a, b, c)
2. bet(b, a, d)    / bet(c, b, d)
--------------------
3. bet(c, b, a)	[B1]
4. bet(c, b, d)	[B5]
🔥 Success! You proved bet(c, b, d) is true.


In [24]:
def zify(template):
    pc = template.split('⊢')   # premises, conclusion
    pc = [ t.strip() for t in pc ]
    return f"Implies(And({', '.join(  list(map( lambda t: parse(t) , pc[0].split(',') ) ) )}), { parse(pc[1]) })"

In [22]:
import re

class TreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

    def __repr__(self):
        OPS = { '¬':'neg', '∧':'land', '∨':'lor', '→':'imp'  }
        """Displays the tree in a clean, human-readable format."""
        if self.left is None and self.right is None:
            return str(self.value)
        if self.right is None:
            return f"neg({self.left})"
        return f"{OPS[self.value]}({self.left}, {self.right})"

    def av(self):
        return f'Valid({self})'

def parse(formula: str)-> TreeNode:
    # Remove all whitespace
    formula = "".join(formula.split()) 
    is_well_formed = True 

    def get_binding_power(operator):
        if operator == '¬':   return [None, 4]  # Right associative
        elif operator == '∧': return [3, 3.1]
        elif operator == '∨': return [2, 2.1]
        elif operator == '→': return [1.1, 1]
        elif operator == '↔': return [0, 0.1]
        else:                 return [0, 0]

    cur_index = -1

    def get_next_token():
        if cur_index + 1 >= len(formula): 
            return False
        return formula[cur_index + 1]

    def parse_formula(min_binding_power=0):  # Default changed to 0
        nonlocal cur_index, is_well_formed 
        cur_index += 1
        
        if cur_index >= len(formula):
            is_well_formed = False
            return None
            
        left = formula[cur_index]
        
        if left.isalpha(): 
            left = TreeNode(left)
        elif left == '(':
            # 1. Parse everything inside with base priority
            left = parse_formula(0) 
            # 2. Force-verify and consume the closing parenthesis right here
            if get_next_token() == ')':
                cur_index += 1 
            else:
                is_well_formed = False  # Missing closing parenthesis!
        elif left == '¬':
            operator = left
            left = parse_formula(get_binding_power(operator)[1])
            node = TreeNode(operator)
            node.left = left
            left = node
        else: 
            is_well_formed = False
            return None

        while True:
            operator = get_next_token()
            if operator is False or operator == ')': 
                break
            elif operator not in ['∧', '∨', '→', '↔']:  
                is_well_formed = False
                return None
                
            left_binding_power = get_binding_power(operator)[0]
            right_binding_power = get_binding_power(operator)[1]
            
            if left_binding_power < min_binding_power:
                break
                
            cur_index += 1 
            operator = formula[cur_index]
            right = parse_formula(right_binding_power)
            
            node = TreeNode(operator)
            node.left = left
            node.right = right
            left = node
            
        return left

    # Run the main parsing loop starting at 0 binding power
    syntax_tree = parse_formula(0)
    
    # Bug 2 Fix: If cur_index didn't reach the last character, trailing garbage exists
    if cur_index < len(formula) - 1:
        is_well_formed = False

    if is_well_formed is False: 
        return False
    return syntax_tree